# Cathy Fit Pipeline

獨立、可重現的三階段星系團紅移 fitting。所有函式集中於一個 cell，執行與分析位於下方。

請先閱讀末尾完整操作說明；預設使用現有 CAMIRA 100 的 LS、Tian、HSC 資料。原始 V2 不會被修改。

## 1. 套件
選用已安裝 numpy / pandas / scipy / astropy / matplotlib / IPython 的 kernel。

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, re, importlib.metadata
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.signal import find_peaks, peak_widths
from scipy.optimize import curve_fit
import astropy.units as u
from astropy.cosmology import FlatLambdaCDM
from astropy.coordinates import SkyCoord, CartesianRepresentation
from IPython.display import display

cosmo = FlatLambdaCDM(H0=70, Om0=0.3)


## 2. 全部函式
讀取與標準化、光譜優先、核心 fitting、三階段執行、繪圖、輸出、跨資料源分析。此 cell 只定義函式，不讀取資料或執行 fitting。

In [ ]:
def select_within_radius(df, center_ra, center_dec, z_cluster, radius_mpc):
    cosmo = FlatLambdaCDM(
    H0=70,
    Om0=0.3
    )
    radius_kpc = radius_mpc * 1000
    radius_arcsec = (radius_kpc * cosmo.arcsec_per_kpc_proper(z_cluster)).value

    c0 = SkyCoord(center_ra*u.deg, center_dec*u.deg)
    cgal = SkyCoord(df["ra"].values*u.deg, df["dec"].values*u.deg)

    sep_arcsec = cgal.separation(c0).arcsec

    out = df.copy()
    out["sep_arcsec"] = sep_arcsec
    out["sep_mpc"] = sep_arcsec / cosmo.arcsec_per_kpc_proper(z_cluster).value / 1000

    return out[out["sep_mpc"] <= radius_mpc].copy()

def postfit_peak_qa(popt, n_components, height_ratio_cut=0.7):
    """
    用 Lorentzian fit 後的 amplitude A 判斷是否有高度太接近的次峰。

    Parameters
    ----------
    popt : array-like
        curve_fit 回傳的最佳參數：
        [A1, z1, gamma1, A2, z2, gamma2, ..., background]

    n_components : int
        Lorentzian component 數量。

    height_ratio_cut : float
        若 second_A / main_A >= 此值，則 drop cluster。

    Returns
    -------
    dict
        包含主峰、次峰資訊與是否 drop。
    """

    result = {
        "drop_cluster": False,
        "drop_reason": None,
        "main_component": np.nan,
        "main_peak_z": np.nan,
        "main_peak_A": np.nan,
        "main_gamma": np.nan,
        "second_component": np.nan,
        "second_peak_z": np.nan,
        "second_peak_A": np.nan,
        "second_gamma": np.nan,
        "second_to_main_ratio": np.nan,
    }

    # fit 不成功或 component 不足時不做雙峰判斷
    if popt is None or n_components < 1:
        result["drop_cluster"] = True
        result["drop_reason"] = "invalid_fit"
        return result

    A_list = np.array([popt[3*i] for i in range(n_components)])
    z_list = np.array([popt[3*i + 1] for i in range(n_components)])
    gamma_list = np.array([popt[3*i + 2] for i in range(n_components)])

    # 有 NaN / inf 就視為 fit 有問題
    if not (
        np.all(np.isfinite(A_list)) and
        np.all(np.isfinite(z_list)) and
        np.all(np.isfinite(gamma_list))
    ):
        result["drop_cluster"] = True
        result["drop_reason"] = "nonfinite_fit_parameter"
        return result

    # A 最大者定義為主峰
    order = np.argsort(A_list)[::-1]

    main_i = order[0]

    result["main_component"] = int(main_i + 1)
    result["main_peak_z"] = z_list[main_i]
    result["main_peak_A"] = A_list[main_i]
    result["main_gamma"] = gamma_list[main_i]

    # 只有一個 component，沒有次峰，不 drop
    if n_components == 1:
        return result

    second_i = order[1]

    result["second_component"] = int(second_i + 1)
    result["second_peak_z"] = z_list[second_i]
    result["second_peak_A"] = A_list[second_i]
    result["second_gamma"] = gamma_list[second_i]

    ratio = A_list[second_i] / A_list[main_i]
    result["second_to_main_ratio"] = ratio

    if ratio >= height_ratio_cut:
        result["drop_cluster"] = True
        result["drop_reason"] = "second_peak_too_high"

    return result

def lorentzian(x, A, x0, gamma):
    return A * gamma**2 / ((x - x0)**2 + gamma**2)

def multi_lorentzian(x, *params):
    y = np.zeros_like(x, dtype=float)
    n_peaks = (len(params) - 1) // 3

    for i in range(n_peaks):
        A = params[3*i]
        x0 = params[3*i + 1]
        gamma = params[3*i + 2]
        y += lorentzian(x, A, x0, gamma)

    return y + params[-1]

def make_redshift_histogram(
    df,
    bins,
    z_col="adopted_z",
    err_col="adopted_z_err",
    weight_by_error=True,
    density=True
):
    """
    Construct either an uncertainty-weighted or unweighted histogram.

    weight_by_error=True:
        weight = 1 / adopted_z_err

    weight_by_error=False:
        every object contributes weight = 1
    """

    z = df[z_col].to_numpy(dtype=float)

    good = np.isfinite(z)

    if weight_by_error:
        zerr = df[err_col].to_numpy(dtype=float)

        good &= (
            np.isfinite(zerr) &
            (zerr > 0)
        )

        z = z[good]
        weights = 1.0 / zerr[good]

    else:
        z = z[good]
        weights = None

    counts, edges = np.histogram(
        z,
        bins=bins,
        weights=weights,
        density=density
    )

    centers = 0.5 * (edges[:-1] + edges[1:])

    return centers, counts, edges, z, weights

def choose_n_components_from_peak_table(row, single_peak_cut=0.6):
    if row.get("single_peak_fraction", 0) > single_peak_cut:
        return 1

    peak_cols = [
        col for col in row.index
        if isinstance(col, (int, np.integer)) and col != 1
    ]

    if len(peak_cols) == 0:
        return 1

    return row[peak_cols].idxmax()

def choose_best_setting_for_cluster(
    cluster,
    n_components,
    test_df,
    sigma_ref=2,
    prominence_ref=0.20
):
    sub = test_df[
        (test_df["cluster_name"] == cluster) &
        (test_df["n_peaks"] == n_components)
    ].copy()

    if len(sub) == 0:
        sub = test_df[test_df["cluster_name"] == cluster].copy()

    sub["distance_to_ref"] = (
        (sub["sigma"] - sigma_ref)**2 +
        ((sub["prominence_frac"] - prominence_ref) / 0.1)**2
    )

    return sub.sort_values("distance_to_ref").iloc[0]

def choose_lorentzian_settings(
    df,
    z_col="adopted_z",
    err_col="adopted_z_err",
    cluster_col="cluster_name",
    z_bins=np.arange(0, 2.01, 0.03),
    sigma_list=(1, 1.5, 2, 2.5, 3),
    prominence_frac_list=(0.10, 0.15, 0.20, 0.25, 0.30),
    single_peak_cut=0.6,
    sigma_ref=2,
    prominence_ref=0.20,
    weight_by_error=True,
    density=True
):
    """
    Test different smoothing and prominence settings for each cluster.

    This function does not perform the final Lorentzian fitting.
    It determines:
        - how many components should be fitted
        - which sigma should be used
        - which prominence fraction should be used
    """

    results = []

    for cluster, group in df.groupby(cluster_col):

        bin_centers, counts, _, z, _ = make_redshift_histogram(
            group,
            bins=z_bins,
            z_col=z_col,
            err_col=err_col,
            weight_by_error=weight_by_error,
            density=density
        )

        if len(z) == 0 or not np.any(np.isfinite(counts)):
            continue

        if np.nanmax(counts) <= 0:
            continue

        for sigma in sigma_list:

            smooth_counts = gaussian_filter1d(
                counts,
                sigma=sigma
            )

            if np.max(smooth_counts) <= 0:
                continue

            for frac in prominence_frac_list:

                prominence_cut = frac * np.max(smooth_counts)

                peaks, props = find_peaks(
                    smooth_counts,
                    prominence=prominence_cut
                )

                results.append({
                    "cluster_name": cluster,
                    "sigma": sigma,
                    "prominence_frac": frac,
                    "prominence_cut": prominence_cut,
                    "n_peaks": len(peaks),
                    "peak_z": bin_centers[peaks],
                    "prominence": props["prominences"]
                })

    test_df = pd.DataFrame(results)

    if test_df.empty:
        return pd.DataFrame(), pd.DataFrame(), pd.DataFrame()

    peak_count_table = (
        test_df
        .groupby(["cluster_name", "n_peaks"])
        .size()
        .unstack(fill_value=0)
    )

    peak_count_table["n_test"] = peak_count_table.sum(axis=1)

    if 1 in peak_count_table.columns:
        peak_count_table["single_peak_fraction"] = (
            peak_count_table[1] /
            peak_count_table["n_test"]
        )
    else:
        peak_count_table["single_peak_fraction"] = 0.0

    peak_count_table["n_components"] = peak_count_table.apply(
        choose_n_components_from_peak_table,
        axis=1,
        single_peak_cut=single_peak_cut
    )

    chosen_settings = []

    for cluster in peak_count_table.index:

        n_components = int(
            peak_count_table.loc[cluster, "n_components"]
        )

        best = choose_best_setting_for_cluster(
            cluster=cluster,
            n_components=n_components,
            test_df=test_df,
            sigma_ref=sigma_ref,
            prominence_ref=prominence_ref
        )

        chosen_settings.append({
            "cluster_name": cluster,
            "n_components": n_components,
            "sigma_use": best["sigma"],
            "prominence_frac_use": best["prominence_frac"],
            "n_peaks_found": best["n_peaks"],
            "peak_z": best["peak_z"],
            "prominence": best["prominence"],
            "weight_by_error": weight_by_error
        })

    chosen_df = pd.DataFrame(chosen_settings)

    return test_df, peak_count_table, chosen_df

def fit_one_cluster_lorentzian(
    group,
    chosen,
    z_col="adopted_z",
    err_col="adopted_z_err",
    cluster_z_col="cluster_zphot",
    z_bins=np.arange(0, 2.01, 0.03),
    height_ratio_cut=0.7,
    weight_by_error=True,
    density=True
):
    """
    Fit one cluster with one or more Lorentzian components.

    Weight and density are explicitly supplied by the stage runner.
    """

    cluster = group["cluster_name"].iloc[0]
    cluster_zphot = group[cluster_z_col].iloc[0]

    n_components_requested = int(chosen["n_components"])
    sigma_use = float(chosen["sigma_use"])
    prominence_frac_use = float(chosen["prominence_frac_use"])

    bin_centers, counts, _, z, _ = make_redshift_histogram(
        group,
        bins=z_bins,
        z_col=z_col,
        err_col=err_col,
        weight_by_error=weight_by_error,
        density=density
    )

    row = {
        "cluster_name": cluster,
        "cluster_zphot": cluster_zphot,
        "n_gal": len(z),
        "n_components_requested": n_components_requested,
        "sigma_use": sigma_use,
        "prominence_frac_use": prominence_frac_use,
        "weight_by_error": weight_by_error
    }

    if len(z) == 0 or np.max(counts) <= 0:
        row.update({
            "n_components": 0,
            "fit_success": False,
            "r2": np.nan,
            "background": np.nan,
            "drop_cluster": True,
            "drop_reason": "no_valid_z"
        })
        return row

    smooth_counts = gaussian_filter1d(
        counts,
        sigma=sigma_use
    )

    peaks, props = find_peaks(
        smooth_counts,
        prominence=(
            prominence_frac_use *
            np.max(smooth_counts)
        )
    )

    if len(peaks) == 0:
        peaks = np.array([np.argmax(smooth_counts)])
        prominences = np.array([np.max(smooth_counts)])
    else:
        prominences = props["prominences"]

    # The number of fitted components cannot exceed available initial peaks
    n_components = min(
        n_components_requested,
        len(peaks)
    )

    order = np.argsort(prominences)[::-1]
    peaks = peaks[order[:n_components]]

    # Sort initial components from low z to high z
    peaks = peaks[np.argsort(bin_centers[peaks])]

    widths = peak_widths(
        smooth_counts,
        peaks,
        rel_height=0.5
    )[0]

    bin_width = bin_centers[1] - bin_centers[0]

    p0 = []
    lower_bounds = []
    upper_bounds = []
    bin_width = np.median(np.diff(z_bins))
    gamma_min = bin_width / 2


    for peak_idx, width in zip(peaks, widths):

        A0 = smooth_counts[peak_idx]
        x00 = bin_centers[peak_idx]
        gamma0 = float(np.clip(width * bin_width / 2, gamma_min + 1e-10, 0.5 - 1e-10))

        p0.extend([A0, x00, gamma0])

        lower_bounds.extend([
            0,
            z_bins[0],
            gamma_min
        ])

        upper_bounds.extend([
            np.inf,
            z_bins[-1],
            0.5
        ])

    # Constant background
    p0.append(0.0)
    lower_bounds.append(0.0)
    upper_bounds.append(np.inf)

    try:
        popt, pcov = curve_fit(
            multi_lorentzian,
            bin_centers,
            smooth_counts,
            p0=p0,
            bounds=(lower_bounds, upper_bounds),
            maxfev=20000
        )

        fit_y = multi_lorentzian(
            bin_centers,
            *popt
        )

        residual = smooth_counts - fit_y
        ss_res = np.sum(residual**2)

        ss_tot = np.sum(
            (
                smooth_counts -
                np.mean(smooth_counts)
            )**2
        )

        r2 = (
            1 - ss_res / ss_tot
            if ss_tot > 0
            else np.nan
        )

        fit_success = True
        fit_error = None

    except Exception as e:

        popt = np.full(len(p0), np.nan)
        pcov = None
        fit_y = np.full_like(smooth_counts, np.nan)
        r2 = np.nan
        fit_success = False
        fit_error = str(e)

    row.update({
        "n_components": n_components,
        "fit_success": fit_success,
        "fit_error": fit_error,
        "r2": r2,
        "background": popt[-1]
    })

    for i in range(n_components):

        A = popt[3*i]
        x0 = popt[3*i + 1]
        gamma = popt[3*i + 2]

        row[f"A{i+1}"] = A
        row[f"z_fit{i+1}"] = x0
        row[f"gamma{i+1}"] = gamma
        row[f"FWHM{i+1}"] = 2 * gamma
        row[f"dz_cluster{i+1}"] = abs(
            x0 - cluster_zphot
        )

    if fit_success:
        qa = postfit_peak_qa(
            popt,
            n_components=n_components,
            height_ratio_cut=height_ratio_cut
        )
    else:
        qa = postfit_peak_qa(
            None,
            n_components=n_components,
            height_ratio_cut=height_ratio_cut
        )
        qa["drop_reason"] = "fit_failed"

    row.update(qa)

    return row

def recenter_members_skycoord(members, weight_col=None):
    if len(members) == 0:
        return np.nan, np.nan

    c = SkyCoord(
        ra=members["ra"].values * u.deg,
        dec=members["dec"].values * u.deg,
        frame="icrs"
    )

    xyz = c.cartesian.xyz.value   # shape = (3, N)

    if weight_col is None:
        weights = np.ones(len(members))
    else:
        weights = members[weight_col].values
        weights = np.where(np.isfinite(weights) & (weights > 0), weights, 0)

        if np.sum(weights) == 0:
            weights = np.ones(len(members))

    mean_xyz = np.average(xyz, axis=1, weights=weights)

    norm = np.linalg.norm(mean_xyz)
    if norm == 0:
        return np.nan, np.nan

    mean_xyz = mean_xyz / norm

    new_coord = SkyCoord(
        CartesianRepresentation(
            mean_xyz[0],
            mean_xyz[1],
            mean_xyz[2]
        ),
        frame="icrs"
    )

    return new_coord.ra.deg, new_coord.dec.deg

def lorentzian_height_interval(
    peak_z,
    gamma,
    height_fraction=0.95
):
    """
    Find the redshift interval where the fitted main Lorentzian
    is higher than a selected fraction of its peak height.

    For a Lorentzian:

        L(z) = A * gamma^2 / ((z - z0)^2 + gamma^2)

    The boundary satisfies:

        L(z) = height_fraction * A

    Parameters
    ----------
    peak_z : float
        Center z0 of the fitted main Lorentzian.

    gamma : float
        Lorentzian HWHM parameter.

    height_fraction : float
        Fraction of the peak height used as the boundary.
        For example, 0.95 means 95% of the maximum height.

    Returns
    -------
    z_low : float
        Lower redshift boundary.

    z_high : float
        Upper redshift boundary.

    z_err : float
        Half-width from the peak to either boundary.

    full_width : float
        Full width of the selected interval.
    """

    if (
        not np.isfinite(peak_z) or
        not np.isfinite(gamma) or
        gamma <= 0 or
        height_fraction <= 0 or
        height_fraction >= 1
    ):
        return np.nan, np.nan, np.nan, np.nan

    # Distance from the peak to the selected height boundary
    z_err = gamma * np.sqrt(
        1.0 / height_fraction - 1.0
    )

    z_low = peak_z - z_err
    z_high = peak_z + z_err

    full_width = z_high - z_low

    return z_low, z_high, z_err, full_width

def compare_angular_scale(
    old_z,
    new_z,
    threshold=0.10
):
    """
    Compare proper kpc per arcsec at old and new redshifts.

    If the fractional change is greater than threshold, use the new z
    when converting 0.5 Mpc to angular radius.
    """

    old_arcsec_per_kpc = (
        cosmo.arcsec_per_kpc_proper(old_z).value
    )

    new_arcsec_per_kpc = (
        cosmo.arcsec_per_kpc_proper(new_z).value
    )

    old_kpc_per_arcsec = 1.0 / old_arcsec_per_kpc
    new_kpc_per_arcsec = 1.0 / new_arcsec_per_kpc

    fractional_difference = abs(
        new_kpc_per_arcsec -
        old_kpc_per_arcsec
    ) / old_kpc_per_arcsec

    use_new_z_for_radius = (
        fractional_difference > threshold
    )

    z_for_radius = (
        new_z
        if use_new_z_for_radius
        else old_z
    )

    return {
        "old_kpc_per_arcsec": old_kpc_per_arcsec,
        "new_kpc_per_arcsec": new_kpc_per_arcsec,
        "scale_fractional_difference": fractional_difference,
        "use_new_z_for_radius": use_new_z_for_radius,
        "z_for_radius": z_for_radius
    }

# ── 1. Standard input, validation, and paired redshift adoption ──────────
def read_csv_columns(path, columns=None):
    """Read only needed columns; identifiers are strings, never floating point."""
    header = pd.read_csv(path, nrows=0).columns
    use = list(header) if columns is None else [c for c in header if c in columns]
    ids = [c for c in use if 'id' in c.lower() or c in {'cluster_name','release'}]
    return pd.read_csv(path, usecols=use, dtype={c:'string' for c in ids})


def valid_pair(z, err):
    return np.isfinite(z) & (z > 0) & np.isfinite(err) & (err > 0)


def normalize_input(config):
    """Adapters -> separate clusters and candidate galaxies. No artificial spec-z."""
    kind = config['format']
    if kind == 'standard':
        clusters = read_csv_columns(config['clusters'])
        galaxies = read_csv_columns(config['galaxies'])
    else:
        photo = {'ls':('ls_photoz','ls_photoz_err'), 'tian':('tian_photoz','tian_photoz_err'),
                 'hsc':('hsc_photoz','hsc_photoz_err')}[kind]
        common = ['cluster_name','cluster_ra','cluster_dec','cluster_zphot','cluster_e_zphot',
                  'camira_bcg_specz','camira_nmem','ra','dec','object_id','release','brickid','objid',
                  'search_radius_deg','desi_z','desi_zerr','desi_zwarn','desi_type',
                  'sdss_z','sdss_zerr','sdss_class','spec_z','spec_z_err','spec_quality_ok']
        spectral_columns=[c for c in pd.read_csv(config['galaxies'],nrows=0).columns if c.startswith(('desi_','sdss_','spec_'))]
        raw = read_csv_columns(config['galaxies'],common + list(photo) + spectral_columns)
        mapping = {'cluster_name':'cluster_id','cluster_ra':'cluster_ra_deg','cluster_dec':'cluster_dec_deg',
                   'cluster_zphot':'cluster_z','cluster_e_zphot':'cluster_z_err',
                   'camira_bcg_specz':'bcg_spec_z','camira_nmem':'catalog_richness'}
        required = ['cluster_name','cluster_ra','cluster_dec','cluster_zphot','ra','dec',*photo]
        missing = set(required)-set(raw)
        if missing: raise ValueError(f"{config['name']}: missing {sorted(missing)}")
        meta = raw[[c for c in mapping if c in raw]].rename(columns=mapping)
        # Repeated metadata must agree; missing vs known is allowed.
        counts = meta.groupby('cluster_id').nunique(dropna=True)
        if (counts > 1).any().any(): raise ValueError('Conflicting cluster metadata in galaxy table')
        clusters = meta.groupby('cluster_id',sort=False,as_index=False).first()
        galaxies = raw.rename(columns={'cluster_name':'cluster_id','ra':'ra_deg','dec':'dec_deg',
                                      photo[0]:'photo_z',photo[1]:'photo_z_err',
                                      'desi_zerr':'desi_z_err','sdss_zerr':'sdss_z_err'})
        if kind != 'hsc':
            for key in ['release','brickid','objid']:
                if key not in raw: raise ValueError(f'Missing LS identity: {key}')
            galaxies['object_id'] = raw[['release','brickid','objid']].astype('string').agg(':'.join,axis=1)
        galaxies['photo_source'] = {'ls':'LS','tian':'Tian','hsc':'DEmP'}[kind]
        galaxies['object_namespace'] = 'HSC' if kind == 'hsc' else 'LS'
        keep = ['cluster_id','object_id','ra_deg','dec_deg','photo_z','photo_z_err','photo_source',
                'object_namespace','search_radius_deg','desi_z','desi_z_err','desi_zwarn','desi_type',
                'sdss_z','sdss_z_err','sdss_class','spec_z','spec_z_err','spec_quality_ok']
        keep += [c for c in galaxies if c.startswith(('desi_','sdss_','spec_')) and c not in keep]
        galaxies = galaxies[[c for c in keep if c in galaxies]].copy()
    for frame, required in [(clusters,['cluster_id','cluster_ra_deg','cluster_dec_deg','cluster_z']),
                             (galaxies,['cluster_id','object_id','ra_deg','dec_deg','photo_z','photo_z_err'])]:
        missing = set(required)-set(frame)
        if missing: raise ValueError(f'Missing standard columns: {sorted(missing)}')
        for key in ['cluster_id','object_id']:
            if key in frame:
                frame[key] = frame[key].astype('string')
                if frame[key].isna().any() or frame[key].str.strip().eq('').any():
                    raise ValueError(f'Missing {key}')
    if clusters['cluster_id'].duplicated().any(): raise ValueError('Duplicate cluster_id')
    if not set(galaxies.cluster_id).issubset(set(clusters.cluster_id)):
        raise ValueError('Galaxy references unknown cluster')
    # Exact repeated rows are collapsed; conflicting source measurements require a decision.
    galaxies = galaxies.drop_duplicates().reset_index(drop=True)
    if galaxies.duplicated(['cluster_id','object_id']).any():
        raise ValueError('Conflicting duplicate cluster_id/object_id')
    for c in ['cluster_ra_deg','cluster_dec_deg','cluster_z','cluster_z_err','bcg_spec_z','catalog_richness']:
        clusters[c] = pd.to_numeric(clusters[c],errors='coerce') if c in clusters else np.nan
    for c in ['bcg_spec_z','cluster_z_err']:
        clusters.loc[~np.isfinite(clusters[c]) | (clusters[c]<=0),c] = np.nan
    for c in ['ra_deg','dec_deg','photo_z','photo_z_err','desi_z','desi_z_err','sdss_z','sdss_z_err','spec_z','spec_z_err']:
        galaxies[c] = pd.to_numeric(galaxies[c],errors='coerce') if c in galaxies else np.nan
    if 'photo_source' not in galaxies: galaxies['photo_source'] = config['name']
    if 'object_namespace' not in galaxies: galaxies['object_namespace'] = 'UNSPECIFIED'
    if clusters.empty: raise ValueError('Cluster table is empty')
    if config.get('spectra'):
        galaxies = attach_spectra(galaxies,config['spectra'])
    galaxies = adopt_redshifts(galaxies)
    return clusters, galaxies


def attach_spectra(galaxies, path):
    """External standardized spectra: exact IDs or <=1 arcsec mutual-nearest coordinates.

    One best reliable spectrum per object per survey must be supplied.
    Each survey is matched independently; matching uses positions, not redshifts.
    Existing in-table measurements win conflicts; conflicts are recorded.
    """
    spec = read_csv_columns(path)
    required = {'spec_z','spec_z_err','spec_source'}
    if not required.issubset(spec): raise ValueError(f'Spectra require {required}')
    for c in ['spec_z','spec_z_err']:
        spec[c] = pd.to_numeric(spec[c],errors='coerce')
    spec['spec_source'] = spec.spec_source.astype(str).str.upper()
    if not spec.spec_source.isin(['DESI','SDSS','SPEC']).all():
        raise ValueError('spec_source must be DESI, SDSS, or SPEC')
    out = galaxies.copy()
    for survey, table in spec.groupby('spec_source',sort=False):
        prefix = survey.lower()
        zcol, ecol = prefix+'_z', prefix+'_z_err'
        table = table.reset_index(drop=True)
        pairs = []
        if {'cluster_id','object_id'}.issubset(table):
            if table.duplicated(['cluster_id','object_id']).any(): raise ValueError('Duplicate spectrum keys')
            joined = out[['cluster_id','object_id']].reset_index().merge(
                table.reset_index().rename(columns={'index':'spec_index'}),
                on=['cluster_id','object_id'],how='inner',validate='one_to_one')
            pairs = [(int(r['index']),int(r['spec_index']),np.nan) for _,r in joined.iterrows()]
        elif {'ra_deg','dec_deg'}.issubset(table):
            for c in ['ra_deg','dec_deg']: table[c]=pd.to_numeric(table[c],errors='coerce')
            if not (np.isfinite(table.ra_deg)&table.ra_deg.between(0,360,inclusive='left')&np.isfinite(table.dec_deg)&table.dec_deg.between(-90,90)).all():
                raise ValueError('Invalid spectrum coordinates')
            for cluster, g in out.groupby('cluster_id'):
                candidates = table.loc[table.cluster_id.eq(cluster)] if 'cluster_id' in table else table
                pairs.extend(mutual_pairs(g,candidates))
        else: raise ValueError('Spectra require IDs or ra_deg/dec_deg')
        for gi, si, sep in pairs:
            row=table.loc[si]
            conflict = np.isfinite(out.at[gi,zcol]) and np.isfinite(row.spec_z) and not np.isclose(out.at[gi,zcol],row.spec_z,rtol=0,atol=1e-8)
            out.at[gi,prefix+'_external_conflict'] = bool(conflict)
            out.at[gi,prefix+'_match_sep_arcsec'] = sep
            # Fill a missing/invalid PAIR as a pair; never combine different spectra.
            if not valid_pair(out.at[gi,zcol],out.at[gi,ecol]):
                out.at[gi,zcol],out.at[gi,ecol] = row.spec_z,row.spec_z_err
                if 'spec_quality_ok' in table:
                    out.at[gi,prefix+'_quality_ok'] = row.spec_quality_ok
    return out


def mutual_pairs(left,right):
    """Return original indices and separations; deterministic one-to-one, <=1 arcsec."""
    valid_l=np.isfinite(left.ra_deg)&left.ra_deg.between(0,360,inclusive='left')&np.isfinite(left.dec_deg)&left.dec_deg.between(-90,90)
    valid_r=np.isfinite(right.ra_deg)&right.ra_deg.between(0,360,inclusive='left')&np.isfinite(right.dec_deg)&right.dec_deg.between(-90,90)
    l,r=left.loc[valid_l],right.loc[valid_r]
    if l.empty or r.empty:return []
    lc=SkyCoord(l.ra_deg.to_numpy()*u.deg,l.dec_deg.to_numpy()*u.deg)
    rc=SkyCoord(r.ra_deg.to_numpy()*u.deg,r.dec_deg.to_numpy()*u.deg)
    j,sep,_=lc.match_to_catalog_sky(rc)
    i,_,_=rc.match_to_catalog_sky(lc)
    good=(sep.arcsec<=1.0)&(i[j]==np.arange(len(l)))
    return [(l.index[a],r.index[j[a]],float(sep.arcsec[a])) for a in np.flatnonzero(good)]


def adopt_redshifts(galaxies):
    """Authoritative cleaned spectrum -> photo-z. Never resurrect rejected pairs.

    Unprocessed inline DESI/SDSS require full quality fields or an explicit
    quality_ok=True assertion. Consistent pairs use the smaller zerr, not a
    fixed DESI priority. Positional catalog cleaning belongs upstream.
    """
    g=galaxies.copy()
    g['adopted_z']=np.nan;g['adopted_z_err']=np.nan;g['z_source']='NONE'
    g['invalid_spec_pair']=False;g['spec_blocked']=False
    policy=g.get('spec_clean_policy',pd.Series('',index=g.index)).fillna('').astype(str)
    processed=policy.ne('')
    if (processed & policy.ne('desi_sdss_available_columns_v1')).any():
        raise ValueError('Unknown clean-spec policy; do not silently ignore provenance')
    status=g.get('spec_clean_status',pd.Series('',index=g.index)).fillna('')
    quality=g.get('spec_quality_ok',pd.Series(False,index=g.index)).astype('string').str.lower().isin(['true','1','1.0'])
    clean_ok=processed & status.eq('matched') & quality & valid_pair(g.spec_z,g.spec_z_err) & g.spec_z_err.lt(.001)
    source=g.get('spec_source',pd.Series('SPEC',index=g.index)).fillna('SPEC')
    clean_ok &= source.isin(['DESI','SDSS','SPEC'])
    g.loc[clean_ok,['adopted_z','adopted_z_err']]=g.loc[clean_ok,['spec_z','spec_z_err']].to_numpy()
    g.loc[clean_ok,'z_source']=source.loc[clean_ok]
    g.loc[processed & status.isin(['redshift_conflict','ambiguous_spec_pair']),'spec_blocked']=True
    g.loc[processed & status.eq('matched') & ~clean_ok,'invalid_spec_pair']=True
    eligible=~processed
    valid={}
    for prefix in ['desi','sdss','spec']:
        good=valid_pair(g[prefix+'_z'],g[prefix+'_z_err']) & g[prefix+'_z_err'].lt(.001)
        q=g.get(prefix+'_quality_ok',pd.Series(pd.NA,index=g.index,dtype='string')).astype('string').str.lower()
        asserted=q.isin(['true','1','1.0'])
        if prefix=='desi':
            fields=['desi_zwarn','desi_type','desi_deltachi2','desi_maskbits','desi_morphtype']
            complete=pd.Series(False,index=g.index)
            if set(fields).issubset(g):
                complete=(pd.to_numeric(g.desi_zwarn,errors='coerce').eq(0)
                    & g.desi_type.astype('string').str.strip().isin(['GALAXY','QSO'])
                    & pd.to_numeric(g.desi_deltachi2,errors='coerce').gt(20)
                    & pd.to_numeric(g.desi_maskbits,errors='coerce').eq(0)
                    & g.desi_morphtype.notna() & g.desi_morphtype.astype('string').str.strip().ne('PSF'))
            good &= (complete | asserted) & (q.isna() | asserted)
        elif prefix=='sdss':
            complete=pd.to_numeric(g.get('sdss_zwarning',pd.Series(np.nan,index=g.index)),errors='coerce').eq(0)
            good &= (complete | asserted) & (q.isna() | asserted)
        else:good &= asserted
        valid[prefix]=(eligible & good).fillna(False)
        g['invalid_spec_pair'] |= eligible & g[prefix+'_z'].notna() & ~good.fillna(False)
    conflict=valid['desi'] & valid['sdss'] & ((g.desi_z-g.sdss_z).abs()>.005+1e-12)
    g.loc[conflict,'spec_blocked']=True
    # Trusted generic spectra are also prevented from rescuing a conflicting pair.
    choices=pd.DataFrame({p:g[p+'_z_err'].where(valid[p]&~conflict,np.inf) for p in ['desi','sdss','spec']})
    best=choices.idxmin(axis=1);has=choices.min(axis=1).lt(np.inf)
    for p,label in [('desi','DESI'),('sdss','SDSS'),('spec','SPEC')]:
        take=has & best.eq(p)
        g.loc[take,['adopted_z','adopted_z_err']]=g.loc[take,[p+'_z',p+'_z_err']].to_numpy()
        g.loc[take,'z_source']=label
    photo=g.z_source.eq('NONE') & valid_pair(g.photo_z,g.photo_z_err)
    g.loc[photo,['adopted_z','adopted_z_err']]=g.loc[photo,['photo_z','photo_z_err']].to_numpy()
    g.loc[photo,'z_source']='PHOTO'
    g['valid_photoz']=np.isfinite(g.photo_z)&g.photo_z.gt(0)
    g['valid_photo_input']=valid_pair(g.photo_z,g.photo_z_err)
    g['valid_position']=np.isfinite(g.ra_deg)&g.ra_deg.between(0,360,inclusive='left')&np.isfinite(g.dec_deg)&g.dec_deg.between(-90,90)
    g['valid_fit_input']=g.valid_position & g.z_source.ne('NONE')
    g['exclusion_reason']=np.select([~g.valid_position,g.z_source.eq('NONE')],['invalid_position','no_valid_redshift_error_pair'],default='')
    return g



# ── 2. Three-stage pipeline. V2 radii, bins, peak search and QA retained ──
def fit_stage(data, center_ra, center_dec, z_radius, radius, weighted, density, bins):
    sub=select_within_radius(data,center_ra,center_dec,z_radius,radius)
    if sub.empty:return sub,None,pd.DataFrame()
    # Redshifts outside histogram are retained in aperture audit, but not histogram.
    inside=sub.adopted_z.between(bins[0],bins[-1])
    if not inside.any():return sub,None,pd.DataFrame()
    test,_,chosen=choose_lorentzian_settings(sub,z_bins=bins,weight_by_error=weighted,density=density)
    if chosen.empty:return sub,None,test
    if int(chosen.iloc[0]['n_components']) < 1:
        fit={'fit_success':False,'drop_cluster':True,'drop_reason':'no_detected_peak',
             'n_components':0,'sigma_use':float(chosen.iloc[0]['sigma_use']),
             'prominence_frac_use':float(chosen.iloc[0]['prominence_frac_use'])}
        return sub,fit,test
    fit=fit_one_cluster_lorentzian(sub,chosen.iloc[0],z_bins=bins,weight_by_error=weighted,density=density)
    return sub,fit,test


def run_pipeline(clusters,galaxies,weighted):
    """No file I/O or notebook globals; return complete audit and plot bundle."""
    if len(weighted)!=3 or any(type(x) is not bool for x in weighted):
        raise ValueError('STAGE_WEIGHTED must contain exactly three bools')
    bins=np.arange(0,2.01,0.03) # V2: last edge is 1.98, not 2.00
    results=[];stages=[];members=[];cache={};search=[]
    groups={k:v for k,v in galaxies.groupby('cluster_id',sort=False)}
    for number,meta in enumerate(clusters.to_dict('records'),1):
        name=meta['cluster_id'];g=groups.get(name,galaxies.iloc[:0])
        row={**meta,'status':'drop','drop_stage':'input','drop_reason':'',
             'z_final':np.nan,'z_peak95_halfwidth':np.nan,'N_peak':np.nan,'N_peak_lowerr':np.nan,
             'N_raw':len(g),'N_photoz':int(g.valid_photoz.sum()),'N_fit_input':int(g.valid_fit_input.sum()),
             **{f'weight_fit{i+1}':w for i,w in enumerate(weighted)}}
        row['N_spec_input']=int((g.valid_fit_input & g.z_source.isin(['DESI','SDSS','SPEC'])).sum())
        z0,ra,dec=meta['cluster_z'],meta['cluster_ra_deg'],meta['cluster_dec_deg']
        if not (np.isfinite(z0) and z0>0 and np.isfinite(ra) and 0<=ra<360 and np.isfinite(dec) and -90<=dec<=90):
            row['drop_reason']='invalid_cluster_metadata';results.append(row);continue
        data=g.loc[g.valid_fit_input].copy()
        data['ra']=data.ra_deg;data['dec']=data.dec_deg
        data['cluster_name']=name;data['cluster_zphot']=z0
        if data.empty:
            row['drop_reason']='no_valid_fit_input';results.append(row);continue
        z_radius=z0
        for stage,(radius,density,w) in enumerate(zip((0.5,0.3,0.5),(True,True,False),weighted),1):
            row['drop_stage']=f'fit{stage}'
            sub,fit,trials=fit_stage(data,ra,dec,z_radius,radius,w,density,bins)
            stage_row={'cluster_id':name,'stage':stage,'weighted':w,'density':density,'radius_mpc':radius,
                       'center_ra_deg':ra,'center_dec_deg':dec,'z_for_radius':z_radius,'N_aperture':len(sub),
                       'N_histogram':int(sub.adopted_z.between(bins[0],bins[-1]).sum()),
                       'median_z_err':float(sub.adopted_z_err.median()) if len(sub) else np.nan,
                       'fit_success':False,'drop_reason':'no_fit_result'}
            if 'search_radius_deg' in g:
                query_radius=pd.to_numeric(g.search_radius_deg,errors='coerce').dropna()
                if len(query_radius):
                    shift=SkyCoord(meta['cluster_ra_deg']*u.deg,meta['cluster_dec_deg']*u.deg).separation(SkyCoord(ra*u.deg,dec*u.deg)).deg
                    angular_radius=(radius*1000*cosmo.arcsec_per_kpc_proper(z_radius)).value/3600
                    stage_row['aperture_within_original_query']=bool(shift+angular_radius<=query_radius.min())
            for source in ['DESI','SDSS','SPEC','PHOTO']:stage_row['N_'+source]=int(sub.z_source.eq(source).sum())
            if fit is not None:stage_row.update(fit)
            stages.append(stage_row)
            cache[(name,stage)]={'data':sub.copy(),'fit':fit,'weighted':w,'density':density}
            if len(trials):
                trials=trials.copy();trials['stage']=stage;search.append(trials)
            selected=sub.copy();selected['stage']=stage
            selected['is_peak_member']=False;selected['is_peak_lowerr_member']=False
            selected['is_recenter_member']=False
            if fit is None or not fit.get('fit_success',False) or fit.get('drop_cluster',True):
                row['drop_reason']=stage_row['drop_reason'];members.append(selected);break
            peak,gamma=fit['main_peak_z'],fit['main_gamma']
            row[f'z_fit{stage}']=peak;row[f'gamma_fit{stage}']=gamma;row[f'fwhm_fit{stage}']=2*gamma
            row[f'r2_fit{stage}']=fit['r2']
            if stage==1:
                # V2 actually used gamma (HWHM), despite old FWHM labels.
                mask=(sub.adopted_z-peak).abs()<=gamma
                selected['is_recenter_member']=mask
                ra,dec=recenter_members_skycoord(sub.loc[mask])
                row.update(final_ra_deg=ra,final_dec_deg=dec)
                members.append(selected)
                if not np.isfinite(ra) or not np.isfinite(dec):
                    row.update(drop_stage='recenter',drop_reason='no_members_for_recenter');break
            elif stage==2:
                members.append(selected)
                if not np.isfinite(peak) or peak<=0:
                    row['drop_reason']='nonpositive_fit2_redshift';break
                scale=compare_angular_scale(z0,peak,threshold=0.10)
                z_radius=scale['z_for_radius'];row.update(scale)
            else:
                lo,hi,half,_=lorentzian_height_interval(peak,gamma,0.95)
                rlo,rhi,_,_=lorentzian_height_interval(peak,gamma,0.7)
                median=float(sub.adopted_z_err.median())
                selected['is_peak_member']=sub.adopted_z.between(rlo,rhi)
                selected['is_peak_lowerr_member']=selected.is_peak_member & sub.adopted_z_err.le(median)
                members.append(selected)
                row.update(status='keep',drop_stage='',drop_reason='',z_final=peak,
                           z_peak95_low=lo,z_peak95_high=hi,z_peak95_halfwidth=half,
                           richness_z_low=rlo,richness_z_high=rhi,final_median_z_err=median,
                           N_peak=int(selected.is_peak_member.sum()),N_peak_lowerr=int(selected.is_peak_lowerr_member.sum()))
        results.append(row)
        if number%20==0 or number==len(clusters):print(f'  fitted {number}/{len(clusters)}')
    result=pd.DataFrame(results)
    for tag,ref in [('catalog','cluster_z'),('bcg','bcg_spec_z')]:
        valid=result.status.eq('keep')&np.isfinite(result[ref])&result[ref].gt(0)
        result['delta_z_'+tag]=(result.z_final-result[ref]).where(valid)
        result['normalized_delta_z_'+tag]=result['delta_z_'+tag]/(1+result[ref])
    membership=pd.concat(members,ignore_index=True) if members else pd.DataFrame(columns=['cluster_id','object_id','stage','is_peak_member','is_peak_lowerr_member'])
    return {'results':result,'stages':pd.DataFrame(stages),'members':membership,'cache':cache,
            'search':pd.concat(search,ignore_index=True) if search else pd.DataFrame(),
            'bins':bins,'clusters':clusters,'galaxies':galaxies,'weighted':weighted}

# ── 3. Plots share the exact histogram convention with fitting ───────────
def save_figure(fig,path,show=False):
    fig.savefig(path,dpi=140,bbox_inches='tight')
    if show:display(fig)
    plt.close(fig)


def plot_cluster(bundle,name,path,show=False):
    row=bundle['results'].set_index('cluster_id').loc[name]
    fig,axes=plt.subplots(1,3,figsize=(18,4.8),constrained_layout=True)
    for stage,ax in enumerate(axes,1):
        entry=bundle['cache'].get((name,stage))
        if entry is None:
            ax.text(.5,.5,'Not reached',ha='center',transform=ax.transAxes);ax.set_title(f'Fit {stage}');continue
        data,fit=entry['data'],entry['fit'];w,density=entry['weighted'],entry['density']
        ax.set(xlabel='Redshift',ylabel=('Weighted PDF' if w else 'PDF') if density else ('Sum of 1/sigma_z per bin' if w else 'Galaxy count per bin'))
        if len(data) and data.adopted_z.between(bundle['bins'][0],bundle['bins'][-1]).any():
            centers,hist,edges,_,_=make_redshift_histogram(data,bundle['bins'],weight_by_error=w,density=density)
            ax.stairs(hist,edges,fill=True,alpha=.25,label='Histogram')
            if fit is not None and 'sigma_use' in fit:
                ax.plot(centers,gaussian_filter1d(hist,fit['sigma_use']),label='Smoothed')
            if fit is not None and fit.get('fit_success',False):
                x=np.linspace(edges[0],edges[-1],1500)
                params=[]
                for i in range(1,int(fit['n_components'])+1):params.extend([fit[f'A{i}'],fit[f'z_fit{i}'],fit[f'gamma{i}']])
                params.append(fit['background']);ax.plot(x,multi_lorentzian(x,*params),'--',label='Lorentzian fit')
                ax.axvline(fit['main_peak_z'],color='tab:red',ls='--',label='Main peak')
                if stage==3:
                    for fraction,color in [(.95,'orange'),(.7,'tab:blue')]:
                        lo,hi,_,_=lorentzian_height_interval(fit['main_peak_z'],fit['main_gamma'],fraction)
                        ax.axvspan(lo,hi,alpha=.12,color=color,label=f'{fraction:.0%} peak-height interval')
        else:ax.text(.5,.5,'No objects within histogram range',ha='center',transform=ax.transAxes)
        ax.axvline(row.cluster_z,color='black',ls=':',label='Catalog z')
        if np.isfinite(row.bcg_spec_z):ax.axvline(row.bcg_spec_z,color='purple',ls=':',label='BCG spec-z')
        status='no fit' if fit is None else (fit.get('drop_reason') or 'passed')
        ax.set_title(f'Fit {stage} | weighted={w} | {status}\nN aperture={len(data)}')
        ax.legend(fontsize=7);ax.grid(alpha=.2)
    fig.suptitle(f"{name} | {row.status} | N_peak={row.N_peak:g}, N_peak_lowerr={row.N_peak_lowerr:g}")
    save_figure(fig,path,show)


def analysis_tables(bundle):
    r=bundle['results'];g=bundle['galaxies']
    quality=g.groupby('cluster_id').agg(N_raw=('object_id','size'),N_photoz=('valid_photoz','sum'),
        N_photo_input=('valid_photo_input','sum'),N_fit_input=('valid_fit_input','sum'),N_invalid_spec=('invalid_spec_pair','sum'))
    sources=pd.crosstab(g.cluster_id,g.z_source).add_prefix('N_adopted_')
    quality=bundle['clusters'][['cluster_id']].merge(quality,on='cluster_id',how='left').merge(sources,on='cluster_id',how='left').fillna(0)
    quality['fraction_photoz']=quality.N_photoz/quality.N_raw.replace(0,np.nan)
    metrics=[]
    for tag in ['catalog','bcg']:
        x=r['normalized_delta_z_'+tag].dropna();delta=r['delta_z_'+tag].dropna()
        metrics.append({'reference':tag,'N':len(x),'median_dz':delta.median(),
            'median_normalized_dz':x.median(),'NMAD':1.4826*(x-x.median()).abs().median(),
            'N_abs_dz_gt_0p2':int(delta.abs().gt(.2).sum())})
    # Quantile bins include all clusters; missing final-z is never converted to zero.
    q=pd.qcut(r.N_photoz,q=5,duplicates='drop') if r.N_photoz.nunique()>1 else pd.Series('all',index=r.index)
    grouped=r.assign(count_bin=q).groupby('count_bin',observed=True)
    drops=grouped.agg(N_min=('N_photoz','min'),N_max=('N_photoz','max'),N_total=('status','size'),
                      N_keep=('status',lambda x:int(x.eq('keep').sum()))).reset_index()
    drops['N_drop']=drops.N_total-drops.N_keep;drops['drop_fraction']=drops.N_drop/drops.N_total
    return quality,pd.DataFrame(metrics),drops


def plot_summary(bundle,path):
    r=bundle['results'];keep=r.loc[r.status.eq('keep')]
    fig,axes=plt.subplots(2,3,figsize=(17,10),constrained_layout=True)
    for ax,ref,label in [(axes[0,0],'cluster_z','Catalog photo-z'),(axes[0,1],'bcg_spec_z','BCG spec-z')]:
        good=keep.loc[np.isfinite(keep[ref])&keep[ref].gt(0)]
        ax.errorbar(good[ref],good.z_final,yerr=good.z_peak95_halfwidth,fmt='o',ms=3,alpha=.65)
        ax.axline((0,0),slope=1,color='black',ls='--')
        ax.set(xlabel=label,ylabel='Final z',title=f'N={len(good)}; bars: 95% peak-height half-width')
    axes[0,2].hist(r.cluster_z.dropna(),bins=20,alpha=.5,label='Catalog (all)')
    axes[0,2].hist(keep.z_final.dropna(),bins=20,alpha=.5,label='Final (keep)')
    axes[0,2].set(xlabel='Redshift',ylabel='Cluster count');axes[0,2].legend()
    counts=r.assign(outcome=np.where(r.status.eq('keep'),'keep',r.drop_stage)).outcome.value_counts()
    axes[1,0].bar(counts.index,counts.values);axes[1,0].tick_params(axis='x',rotation=30)
    axes[1,0].set(ylabel='Cluster count',title='Keep / drop stage')
    axes[1,1].scatter(keep.N_peak,keep.N_peak_lowerr,s=20);axes[1,1].axline((0,0),slope=1,color='black',ls='--')
    axes[1,1].set(xlabel='N_peak',ylabel='N_peak_lowerr',title='Two provisional richness references')
    for col in ['N_peak','N_peak_lowerr']:
        axes[1,2].scatter(keep.catalog_richness,keep[col],s=15,alpha=.6,label=col)
    axes[1,2].set(xlabel='Catalog richness',ylabel='Our candidate count');axes[1,2].legend()
    for ax in axes.flat:ax.grid(alpha=.2)
    save_figure(fig,path,show=True)


def export_run(bundle,directory,config):
    """Save all numerical information needed to audit/replot without fitting."""
    directory=Path(directory);directory.mkdir(parents=True,exist_ok=False)
    for key,file in [('results','cluster_results'),('stages','stage_results'),('members','stage_members'),
                     ('search','peak_search'),('clusters','clusters_standard'),('galaxies','galaxies_standard')]:
        frame=bundle[key].copy()
        for c in frame:
            if frame[c].dtype==object:
                frame[c]=frame[c].map(lambda x:json.dumps(x.tolist()) if isinstance(x,np.ndarray) else x)
        frame.to_csv(directory/f'{file}.csv',index=False)
    quality,metrics,drops=analysis_tables(bundle)
    quality.to_csv(directory/'data_quality.csv',index=False);metrics.to_csv(directory/'redshift_metrics.csv',index=False)
    drops.to_csv(directory/'count_drop_bins.csv',index=False)
    r=bundle['results'];r.loc[r.delta_z_catalog.abs().gt(.2)].to_csv(directory/'outliers.csv',index=False)
    m=bundle['members'];final=m.loc[m.stage.eq(3)].copy()
    final.to_csv(directory/'final_aperture_objects.csv',index=False)
    final.loc[final.is_peak_member].to_csv(directory/'N_peak_members.csv',index=False)
    final.loc[final.is_peak_lowerr_member].to_csv(directory/'N_peak_lowerr_members.csv',index=False)
    histograms=[]
    for (name,stage),entry in bundle['cache'].items():
        d=entry['data']
        if not len(d) or not d.adopted_z.between(bundle['bins'][0],bundle['bins'][-1]).any():continue
        centers,hist,edges,_,_=make_redshift_histogram(d,bundle['bins'],weight_by_error=entry['weighted'],density=entry['density'])
        h=pd.DataFrame({'cluster_id':name,'stage':stage,'bin_left':edges[:-1],'bin_right':edges[1:],
                        'bin_center':centers,'histogram':hist,'weighted':entry['weighted'],'density':entry['density']})
        histograms.append(h)
    (pd.concat(histograms,ignore_index=True) if histograms else pd.DataFrame(columns=['cluster_id','stage'])).to_csv(directory/'histograms.csv',index=False)
    manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'dataset':config,
        'stage_weighted':bundle['weighted'],'stage_density':[True,True,False],
        'radii_mpc':[.5,.3,.5],'z_bins':bundle['bins'].tolist(),'H0':70,'Om0':.3,
        'sigma_grid':[1,1.5,2,2.5,3],'prominence_fraction_grid':[.10,.15,.20,.25,.30],
        'single_peak_fraction_cut':.6,'sigma_reference':2,'prominence_reference':.2,
        'gamma_bounds':[.015,.5],'weight_formula':'1 / adopted_z_err',
        'recenter_interval':'main peak +/- gamma (HWHM)','scale_threshold':.1,'secondary_peak_ratio_cut':.7,
        'spec_policy':'cleaned spectrum (minimum-error consistent DESI/SDSS pair) -> photo-z',
        'invalid_spec_pair_policy':'conflicting/ambiguous clean pairs cannot be rescued by raw spectroscopy',
        'spec_error_strict_max':.001,'spec_conflict_delta_z_max':.005,
        'richness_interval_peak_height':.7,'lowerr_rule':'<= median error of all valid final-aperture objects',
        'z_error_definition':'95% peak-height half-width; NOT confidence interval',
        'versions':{m:importlib.metadata.version(m) for m in ['numpy','pandas','scipy','astropy','matplotlib']},
        'input_files':{str(p):{'bytes':Path(p).stat().st_size,'mtime_ns':Path(p).stat().st_mtime_ns}
                       for p in [config.get('galaxies'),config.get('clusters'),config.get('spectra')] if p}}
    (directory/'run_manifest.json').write_text(json.dumps(manifest,indent=2,ensure_ascii=False,default=str))
    counts=r.status.value_counts().to_dict()
    summary=f"# {config['name']}\n\nClusters: {len(r)}; status: {counts}\n\nWeighted: {bundle['weighted']}\n\nAdopted spectra in input: {int(r.N_spec_input.sum())}\n\nN_peak and N_peak_lowerr are provisional counts, without background subtraction.\n"
    if r.N_spec_input.sum()==0:summary+='\nNo usable individual-galaxy spectroscopy was present. BCG spec-z is reference only.\n'
    (directory/'summary.md').write_text(summary)
    return directory


def load_run(directory):
    """Reconstruct plotting bundle exclusively from exported files (no pickle)."""
    d=Path(directory);manifest=json.loads((d/'run_manifest.json').read_text())
    frames={}
    for key,file in [('results','cluster_results'),('stages','stage_results'),('members','stage_members'),('clusters','clusters_standard'),('galaxies','galaxies_standard')]:
        try:frames[key]=read_csv_columns(d/f'{file}.csv')
        except pd.errors.EmptyDataError:frames[key]=pd.DataFrame()
    frames.update(bins=np.asarray(manifest['z_bins']),weighted=tuple(manifest['stage_weighted']),cache={})
    for record in frames['stages'].to_dict('records'):
        name,stage=record['cluster_id'],int(record['stage'])
        data=frames['members'].loc[frames['members'].cluster_id.eq(name)&frames['members'].stage.eq(stage)]
        fit=record if bool(record['fit_success']) or np.isfinite(record.get('sigma_use', np.nan)) else None
        if fit and pd.isna(fit.get('drop_reason')):fit['drop_reason']=None
        frames['cache'][(name,stage)]={'data':data,'fit':fit,'weighted':bool(record['weighted']),'density':bool(record['density'])}
    return frames


def plot_count_relations(bundles,path):
    names=list(bundles);fig,axes=plt.subplots(3,len(names),figsize=(5*len(names),12),squeeze=False,constrained_layout=True)
    for j,name in enumerate(names):
        b=bundles[name];r=b['results']
        for i,tag in enumerate(['catalog','bcg']):
            axes[i,j].scatter(r.N_photoz,r['normalized_delta_z_'+tag].abs(),s=18,alpha=.65)
            axes[i,j].set(xlabel='Valid photo-z in native input',ylabel='|delta z| / (1 + z_ref)',title=f'{name}: {tag}')
        _,_,drops=analysis_tables(b)
        x=np.arange(len(drops));ax=axes[2,j]
        ax.bar(x,drops.N_keep,label='keep');ax.bar(x,drops.N_drop,bottom=drops.N_keep,label='drop')
        ax.set_xticks(x,labels=[f'{a}-{c}' for a,c in zip(drops.N_min,drops.N_max)],rotation=30)
        ax.set(xlabel='Native photo-z count bin',ylabel='Cluster count',title=name);ax.legend()
        twin=ax.twinx();twin.plot(x,drops.drop_fraction,'ko-');twin.set(ylabel='Drop fraction',ylim=(0,1))
    save_figure(fig,path,show=True)


# ── 4. Cross-survey analysis: native counts and matched objects separated ─
def compare_datasets(bundles,directory):
    """Only identical cluster IDs are compared; source IDs or mutual 1\" matches."""
    directory=Path(directory);directory.mkdir(parents=True,exist_ok=True)
    results=pd.concat([b['results'].assign(dataset=n) for n,b in bundles.items()],ignore_index=True)
    results.to_csv(directory/'all_dataset_results.csv',index=False)
    plot_count_relations(bundles,directory/'counts_accuracy_drop.png')
    if len(bundles)<2:return
    names=list(bundles);stats=[];pair_tables=[]
    for ai,a in enumerate(names):
        for b in names[ai+1:]:
            left=bundles[a]['galaxies'];right=bundles[b]['galaxies']
            namespace_a=set(left.object_namespace.dropna());namespace_b=set(right.object_namespace.dropna())
            same_identity=(len(namespace_a)==1 and namespace_a==namespace_b and 'UNSPECIFIED' not in namespace_a)
            matched=[]
            for cluster in sorted(set(left.cluster_id)&set(right.cluster_id)):
                l=left.loc[left.cluster_id.eq(cluster)];r=right.loc[right.cluster_id.eq(cluster)]
                if same_identity:
                    keys=l.reset_index().merge(r.reset_index(),on='object_id',suffixes=('_a','_b'))
                    pairs=[(x.index_a,x.index_b,np.nan) for x in keys.itertuples()]
                else:pairs=mutual_pairs(l,r)
                valid_a=0;valid_b=0
                if pairs:
                    li,ri,seps=zip(*pairs)
                    x,y=left.loc[list(li)],right.loc[list(ri)]
                    matched.append(pd.DataFrame({'cluster_id':cluster,
                        'object_id_a':x.object_id.to_numpy(),'object_id_b':y.object_id.to_numpy(),
                        'photo_z_a':x.photo_z.to_numpy(),'photo_z_b':y.photo_z.to_numpy(),
                        'separation_arcsec':seps,'dataset_a':a,'dataset_b':b}))
                    valid_a=int(x.valid_photoz.sum());valid_b=int(y.valid_photoz.sum())
                stats.append({'dataset_a':a,'dataset_b':b,'cluster_id':cluster,'N_raw_a':len(l),'N_raw_b':len(r),
                    'N_matched':len(pairs),'N_matched_valid_a':valid_a,'N_matched_valid_b':valid_b,
                    'fraction_b_over_raw_a':valid_b/len(l) if len(l) else np.nan})
            pairs=pd.concat(matched,ignore_index=True) if matched else pd.DataFrame()
            if pairs.empty:continue
            pair_tables.append(pairs)
            pairs.to_csv(directory/f'{safe_name(a)}__{safe_name(b)}_matches.csv',index=False)
            unique=pairs.drop_duplicates(['object_id_a','object_id_b'])
            valid=unique.loc[np.isfinite(unique.photo_z_a)&unique.photo_z_a.gt(0)&np.isfinite(unique.photo_z_b)&unique.photo_z_b.gt(0)]
            fig,ax=plt.subplots(figsize=(6,6))
            if len(valid):ax.hexbin(valid.photo_z_a,valid.photo_z_b,gridsize=70,mincnt=1,bins='log')
            else:ax.text(.5,.5,'No valid paired photo-z',ha='center',transform=ax.transAxes)
            ax.axline((0,0),slope=1,color='black',ls='--');ax.set(xlabel=a+' photo-z',ylabel=b+' photo-z',title=f'Unique valid pairs: {len(valid)}')
            save_figure(fig,directory/f'{safe_name(a)}__{safe_name(b)}_photoz.png')
            comp=bundles[a]['results'].merge(bundles[b]['results'],on='cluster_id',suffixes=('_a','_b'),validate='one_to_one')
            comp.to_csv(directory/f'{safe_name(a)}__{safe_name(b)}_cluster_comparison.csv',index=False)
            fig,axes=plt.subplots(1,3,figsize=(15,4.5),constrained_layout=True)
            for ax,col in zip(axes,['z_final','N_peak','N_peak_lowerr']):
                ax.scatter(comp[col+'_a'],comp[col+'_b'],s=16);ax.axline((0,0),slope=1,color='black',ls='--')
                ax.set(xlabel=a+' '+col,ylabel=b+' '+col)
            save_figure(fig,directory/f'{safe_name(a)}__{safe_name(b)}_cluster_comparison.png')
    pd.DataFrame(stats).to_csv(directory/'match_completeness.csv',index=False)
    # Common LS denominator only for datasets with the same cluster coverage.
    base=next((n for n in names if bundles[n]['galaxies'].photo_source.eq('LS').all()),None)
    if base:
        g=bundles[base]['galaxies'];table=g.groupby('cluster_id').agg(N_raw_LS=('object_id','size'),N_valid_LS=('valid_photoz','sum')).reset_index()
        table['fraction_LS']=table.N_valid_LS/table.N_raw_LS
        for n in names:
            if n==base:continue
            relevant=[p for p in pair_tables if {p.dataset_a.iloc[0],p.dataset_b.iloc[0]}=={base,n}]
            if not relevant:continue
            p=relevant[0];col='photo_z_b' if p.dataset_a.iloc[0]==base else 'photo_z_a'
            count=p.assign(valid=np.isfinite(p[col])&p[col].gt(0)).groupby('cluster_id').valid.sum()
            covered=table.cluster_id.isin(bundles[n]['clusters'].cluster_id)
            table['N_valid_'+n]=table.cluster_id.map(count).fillna(0).where(covered)
            table['fraction_'+n]=table['N_valid_'+n]/table.N_raw_LS
        table.to_csv(directory/'common_LS_denominator_completeness.csv',index=False)
        fractions=[c for c in table if c.startswith('fraction_')]
        ax=table.set_index('cluster_id')[fractions].plot(figsize=(16,5),marker='.',ls='',rot=90)
        ax.set(ylabel='Valid photo-z / original LS objects',ylim=(0,1.05))
        save_figure(ax.figure,directory/'common_LS_denominator_completeness.png')
    # Same deterministic five clusters, all native data (not matched subset).
    common=set.intersection(*(set(bundles[n]['clusters'].cluster_id) for n in names))
    sample=pd.Series(sorted(common),dtype='string').sample(n=min(5,len(common)),random_state=42).tolist()
    for cluster in sample:
        fig,ax=plt.subplots(figsize=(9,5))
        data=[bundles[n]['galaxies'].loc[lambda x:x.cluster_id.eq(cluster)&x.valid_photoz,'photo_z'].to_numpy() for n in names]
        nonempty=[x for x in data if len(x)]
        if nonempty:
            edges=np.histogram_bin_edges(np.concatenate(nonempty),bins=70)
            for n,z in zip(names,data):ax.hist(z,bins=edges,histtype='step',label=n)
        ax.set(xlabel='Photo-z (native catalog)',ylabel='Object count',title=cluster);ax.legend()
        save_figure(fig,directory/f'{safe_name(cluster)}_native_distributions.png')


def safe_name(value):
    return re.sub(r'[^A-Za-z0-9_.+-]+','_',str(value))


## 3. 執行設定
發布版預設使用 examples/ 的合成 photo-z 資料，僅供確認流程可執行，不是科學驗證。分析真實資料時，依 README.md 修改 DATA_ROOT 與 DATASETS。三階段 weighted 仍是唯一方法開關。


In [ ]:
# 請從 repository 根目錄啟動 Jupyter。
DATA_ROOT = Path.cwd() / 'examples'
OUTPUT_ROOT = Path.cwd() / 'cathy_pipeline_outputs'
STAGE_WEIGHTED = (True, True, False)
DATASETS = [
    {'name': 'Synthetic', 'format': 'standard',
     'clusters': DATA_ROOT / 'clusters.csv',
     'galaxies': DATA_ROOT / 'galaxies.csv', 'spectra': None},
]
RUN_DIR = OUTPUT_ROOT / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
RUN_DIR.mkdir(parents=True, exist_ok=False)
print('Output:', RUN_DIR)


## 4. 讀取、驗證與採用紅移
先採用上游清理完成的 DESI/SDSS spec-z 與同來源 error，否則使用 photo-z。已排除的衝突配對不會被 raw 光譜欄位重新採用。


In [ ]:
prepared = {}
for config in DATASETS:
    clusters, galaxies = normalize_input(config)
    prepared[config['name']] = (clusters, galaxies)
    print(config['name'], 'clusters=',len(clusters),'objects=',len(galaxies))
    display(galaxies.z_source.value_counts().rename('N_adopted'))
    if not galaxies.z_source.isin(['DESI','SDSS','SPEC']).any():
        print('No valid individual-galaxy spectra: this run uses available photo-z only.')


## 5. 三階段 fitting 與數值輸出
每個資料源使用相同 weighted 設定。失敗或零有效輸入的 cluster 仍保留在總表。

In [ ]:
bundles = {}
run_paths = {}
for config in DATASETS:
    name = config['name']
    print('Fitting:', name)
    bundle = run_pipeline(*prepared[name], STAGE_WEIGHTED)
    bundles[name] = bundle
    weight_tag = ''.join(str(int(w)) for w in STAGE_WEIGHTED)
    run_paths[name] = export_run(bundle, RUN_DIR/f'{safe_name(name)}_W{weight_tag}', config)
    display(bundle['results'].status.value_counts().rename('N_clusters'))
    display(bundle['results'][['cluster_id','status','drop_stage','drop_reason','z_final','z_peak95_halfwidth','N_peak','N_peak_lowerr']].head(10))


## 6. 摘要、全部 fitting 圖、診斷名單
每個 cluster 都存圖，包括 drop。notebook 展示每份資料 keep 中隨機一例；CSV 索引列出隨機五個、最大偏差六個與有效 photo-z 數最多五個。兩種 richness 只作參考，沒有背景扣除。

In [ ]:
for name, bundle in bundles.items():
    directory = run_paths[name]
    plot_summary(bundle, directory/'summary.png')
    fit_dir = directory/'fits'
    fit_dir.mkdir(exist_ok=True)
    result = bundle['results']
    keep = result.loc[result.status.eq('keep')]
    random_names = keep.sample(n=min(5,len(keep)),random_state=42).cluster_id.tolist()
    selected = []
    for label, frame in [
        ('random_keep',result.loc[result.cluster_id.isin(random_names)]),
        ('outlier',keep.loc[keep.delta_z_catalog.abs().gt(.2)].assign(abs_dz=lambda x:x.delta_z_catalog.abs()).nlargest(6,'abs_dz')),
        ('top_photoz_count',result.nlargest(5,'N_photoz')),
    ]:
        selected.append(frame.assign(diagnostic_group=label))
    pd.concat(selected,ignore_index=True).to_csv(directory/'diagnostic_selection.csv',index=False)
    for cluster in result.cluster_id:
        plot_cluster(bundle,cluster,fit_dir/f'{safe_name(cluster)}.png',show=bool(random_names and cluster==random_names[0]))


## 7. 多資料源比較與物件數分析
配對使用相同 ID 或 ≤1 arcsec 互為最近鄰。原生樣本數與共同 LS 分母分開輸出；單一資料源時仍產生物件數與偏差/drop 分析。

In [ ]:
compare_datasets(bundles, RUN_DIR/'comparison')
print('All results saved to:', RUN_DIR)


完整操作與科學定義見 [Cathy_fit_pipeline_guide.md](Cathy_fit_pipeline_guide.md)。發布版預設合成資料的操作方式見 [README.md](README.md)。
